# Investigação forense da apuração presidencial — 2026

## O que os dados mostram? Existe evidência de fraude?

**Um laboratório de investigação aberto ao público, escrito para ser compreendido sem conhecimento de programação.**

A pergunta não é se uma curva *parece estranha*. Queremos descobrir se há evidências verificáveis de **alteração, acréscimo, omissão ou contabilização indevida de votos**.

Para isso, vamos separar três coisas que frequentemente são misturadas:

1. **Votos registrados nas urnas** — boletins individuais (BU/BWEB).
2. **Horário em que esses registros aparecem como recebidos** — `DT_BU_RECEBIDO`.
3. **Horário em que os totais aparecem publicados ao público** — série nacional EA20.

Uma diferença entre (2) e (3) pode significar processamento, fila de divulgação, diferenças de relógio ou algo que exige investigação. **Não é, por si só, uma alteração de votos.**

Este painel usa resultados locais que já passaram por verificações de integridade no projeto TSE. Não inventa dados, não procura apenas resultados favoráveis à hipótese de fraude e não confunde compatibilidade matemática com comprovação de um mecanismo real.

**Como ler:** cada figura traz uma pergunta popular, a observação, a explicação alternativa e o que ainda precisamos conferir. Os gráficos são exportados como PNG para compartilhar, acompanhados das ressalvas correspondentes.

*Escopo: eleição presidencial, primeiro turno de 4 de outubro de 2026. Análise independente. Nem o TSE nem terceiros endossam automaticamente estas interpretações.*

## Perguntas e critérios de prova

| Pergunta do público | Evidência que realmente a responderia |
|---|---|
| **Os votos foram alterados?** | Divergência reproduzível, por seção e candidato, entre BU autenticado e total oficial correspondente |
| **Votos foram adicionados ou omitidos?** | Contagens por seção que não reconciliam com o conjunto correto de BUs, após esclarecer eleições, locais no exterior e votos anulados |
| **A divulgação foi artificialmente controlada?** | Contradição documentada entre horários, eventos e registros operacionais, com relógios e semânticas certificados |
| **A linha reta da apuração prova fraude?** | Não. Precisamos de testes contra mecanismos normais e de evidência adicional que diferencie as hipóteses |

**Procurar fraude não significa assumir que houve fraude.** Significa formular testes capazes de detectar adulteração caso ela exista — e admitir quando os dados disponíveis não conseguem decidir.

## 1 — Carregar as evidências e conferir sua integridade

Os gráficos a seguir são gerados a partir de **CSV e JSON já produzidos no seu próprio computador** pelas fases V, W, X e Y. Arquivos inexistentes serão marcados como pendentes; nenhum valor será inventado. Cada CSV de saída é novamente verificado por SHA-256.

In [ ]:
from pathlib import Path
import sys, json, hashlib, math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def locate_repo():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p/"start_windows.bat").exists():
            return p
    raise FileNotFoundError("Abra o painel no Jupyter iniciado na pasta TSE.")

REPO=locate_repo()
sys.path.insert(0,str(REPO))
from urnas import root_path
BASE=root_path(None)/"analysis"/"periodicidade_bweb_2026"
EXPORT=BASE/"divulgacao_publica_2026"
EXPORT.mkdir(parents=True,exist_ok=True)

NAVY="#183153"
TEAL="#197A74"
AMBER="#CF8540"
RED="#B94F4D"
GREY="#6B7280"
LIGHT="#E6EDF2"
plt.rcParams.update({
    "figure.figsize":(11,5),"figure.dpi":115,
    "savefig.dpi":190,
    "font.size":11,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.labelcolor":NAVY,"text.color":NAVY,
    "axes.titleweight":"bold",
})

def digest(path):
    sha=hashlib.sha256()
    with path.open("rb") as fp:
        for chunk in iter(lambda:fp.read(4*1024*1024),b""):
            sha.update(chunk)
    return sha.hexdigest()

def summary_if_available(filename,expected_status):
    path=BASE/filename
    if not path.is_file():
        display(Markdown("**Pendente:** relatório `"+filename+"` não foi gerado neste computador."))
        return None
    d=json.loads(path.read_text(encoding="utf-8"))
    if d.get("status")!=expected_status:
        raise RuntimeError("Status inesperado em "+filename)
    if d.get("year")!=2026 or d.get("round")!=1:
        raise RuntimeError("Arquivo de outra eleição/turno: "+filename)
    return d

def checked_csv(meta):
    path=Path(meta["path"])
    if not path.is_file():
        path=BASE/path.name
    if not path.is_file() or digest(path)!=meta["sha256"]:
        raise RuntimeError("SHA-256 não confere em "+str(path))
    return pd.read_csv(path,sep=";",encoding="utf-8-sig",
                       dtype={"UF":str,"municipio":str,"zona":str})

def publish(fig,name):
    fig.tight_layout()
    fig.savefig(EXPORT/(name+".png"),dpi=190,facecolor="white",bbox_inches="tight")
    plt.show()

v=summary_if_available("phase5q_zone_holdout_2026_summary.json",
                       "ZONA_PRE_CORTE_TESTADA_NA_CAUDA_BWEB_2026")
w=summary_if_available("phase5r_cutoff_placebos_2026_summary.json",
                       "PLACEBOS_TEMPORAIS_INTRA_ZONA_2026")
x=summary_if_available("phase5s_ea20_publication_mechanics_2026_summary.json",
                       "MECANISMO_DA_CURVA_PUBLICADA_EA20_2026")
y=summary_if_available("phase5t_receipt_vs_publication_2026_summary.json",
                       "BWEB_RECEBIMENTO_VS_PUBLICACAO_EA20_2026")

if x:
    actual_ea20=REPO/"data/forensics/analysis/phase5a_presidential_national_2026.csv"
    if digest(actual_ea20)!=x["source_sha256"]["EA20_national"]:
        raise RuntimeError("SHA da série nacional EA20 não confere")
print("Pasta de figuras:",EXPORT)
print("Fases encontradas:",", ".join(n for n,d in [("V",v),("W",w),("X",x),("Y",y)] if d) or "nenhuma")

## 2 — Por que a apuração parece uma linha reta?

**Pergunta:** uma linha acumulada muito lisa seria impossível sem interferência?

O gráfico de cima mostra a participação nacional **acumulada** de Lula na cauda após as 20h04. A linha tracejada é um experimento: mantém exatamente o volume de votos de cada atualização, mas dá a todos os lotes a mesma proporção que só conhecemos **ao final**.

**Importante:** esse experimento é retrospectivo; ele não pode ser apresentado como previsão independente. A escala vertical está ampliada para tornar visível uma variação pequena em pontos percentuais.

In [ ]:
if x:
    b=checked_csv(x["output_files"]["batches"])
    start=x["at_cutoff"]
    base_l=100*start["lula"]/start["valid"]
    b0=pd.DataFrame([{"cum_sections":start["sections"],"actual_cum_lula_pct":base_l,
                       "oracle_cum_lula_pct":base_l}])
    curve=pd.concat([b0,b[["cum_sections","actual_cum_lula_pct","oracle_cum_lula_pct"]]],
                    ignore_index=True)
    fig,ax=plt.subplots(figsize=(11,5))
    ax.plot(curve["cum_sections"],curve["actual_cum_lula_pct"],
            color=TEAL,lw=2.8,label="Apuração realmente publicada")
    ax.plot(curve["cum_sections"],curve["oracle_cum_lula_pct"],
            color=AMBER,lw=2.3,ls="--",
            label="Controle: proporção constante (conhecendo o final)")
    ax.set(title="Uma linha muito lisa também pode surgir da própria acumulação",
           xlabel="Seções contabilizadas na divulgação nacional (EA20)",
           ylabel="Participação acumulada de Lula (%)")
    ax.legend(loc="lower right",frameon=False)
    ax.grid(axis="y",alpha=.16)
    ax.text(.015,.025,"Eixo vertical ampliado • controle retrospectivo, não prova de mecanismo real",
            transform=ax.transAxes,color=GREY,fontsize=9)
    publish(fig,"01_linha_acumulada_e_controle")
    display(Markdown(
        "**O que observamos:** a curva publicada é muito regular, mas uma simulação "
        "que conserva os volumes publicados produz outra ainda mais regular. "
        "**O que isso significa:** a aparência de linha reta, sozinha, não permite "
        "identificar adulteração. **O que falta:** confirmar como os BUs foram efetivamente "
        "agrupados e publicados pelo sistema."
    ))
else:
    display(Markdown("Execute a **opção X** para construir esta figura diretamente dos resultados verificados."))

## 3 — O que a linha reta esconde? Quatro grupos de votos diferentes

O resultado acumulado pode mudar pouco porque já há dezenas de milhões de votos no denominador. Por isso o comportamento dos **novos votos de cada lote** é mais revelador do que olhar apenas para a média acumulada.

Aqui os 18,8 milhões de votos publicados depois do corte são divididos em **quatro grupos consecutivos de tamanho aproximado**. Cada barra começa em zero e tem a mesma escala (0–100%); candidatos menores aparecem como "demais".

In [ ]:
if x:
    blocks=checked_csv(x["output_files"]["blocks"]).sort_values("block")
    labels=["Bloco "+str(int(z)) for z in blocks["block"]]
    lula=blocks["lula_pct"].to_numpy()
    flavio=blocks["flavio_pct"].to_numpy()
    others=100-lula-flavio
    idx=np.arange(len(blocks))
    fig,ax=plt.subplots(figsize=(11,5))
    ax.barh(idx,lula,color=TEAL,label="Lula")
    ax.barh(idx,flavio,left=lula,color=AMBER,label="Flávio Bolsonaro")
    ax.barh(idx,others,left=lula+flavio,color=LIGHT,label="Demais votos válidos")
    ax.set_yticks(idx,labels)
    ax.invert_yaxis()
    ax.set_xlim(0,100)
    ax.set(xlabel="% dos votos válidos acrescentados em cada bloco",
           title="A linha acumulada escondia mudanças reais na composição dos lotes")
    ax.legend(ncol=3,loc="lower center",bbox_to_anchor=(.5,-.32),frameon=False)
    for i,(l,valids) in enumerate(zip(lula,blocks["valid"])):
        ax.text(max(5,l/2),i,f"{l:.1f}% Lula",va="center",ha="center",
                color="white",weight="bold")
        ax.text(101,i,f"{valids/1_000_000:.2f} mi de votos",va="center",
                fontsize=9,color=GREY)
    ax.set_xlim(0,119)
    publish(fig,"02_quatro_blocos_de_votos")
    display(Markdown("**O que observamos:** entre os quatro grupos de votos "
        "publicados, a participação de Lula muda consideravelmente. "
        "**Explicação alternativa:** regiões e seções com perfis diferentes "
        "podem chegar em horários diferentes. "
        "**Pergunta ainda aberta:** os BUs correspondentes explicam esses quatro blocos "
        "ou existe uma divergência de contabilização?"))
else:
    display(Markdown("Execute a **opção X** para gerar a comparação dos quatro blocos."))

## 4 — Os votos recebidos mais tarde eram diferentes dentro das mesmas zonas?

**Pergunta:** a mudança depois das 20h04 aconteceu de repente naquele horário?

A comparação das fases V/W usa votos de BUs **recebidos** antes e depois de dez horários predefinidos. Consideramos somente zonas com pelo menos 1.000 votos válidos de cada lado de cada corte.

Atenção: **as zonas elegíveis mudam entre horários**. O aumento às 21h, por exemplo, representa um conjunto muito menor. Isso impede uma interpretação como experimento controlado.

In [ ]:
if w:
    cut=checked_csv(w["output_files"]["cutoffs"])
    cut["clock"]=pd.to_datetime(cut["cutoff_brt"])
    fig,(a1,a2)=plt.subplots(2,1,figsize=(11,7),sharex=True,
                             gridspec_kw={"height_ratios":[2,1]})
    a1.plot(cut["clock"],cut["weighted_shift_lula_pp"],color=TEAL,
            lw=2.5,marker="o",markersize=6)
    a1.axhline(0,color=GREY,lw=1,ls=":")
    a1.axvline(pd.Timestamp("2026-10-04 20:04:39"),color=RED,lw=1.5,ls="--")
    a1.set(ylabel="Lula: recebido depois − antes (pontos percentuais)",
           title="A diferença já existia antes das 20h04")
    a1.grid(axis="y",alpha=.17)
    a1.annotate("Corte investigado\n20h04min39s",
                xy=(pd.Timestamp("2026-10-04 20:04:39"),
                    float(cut.loc[cut["cutoff_brt"]=="2026-10-04 20:04:39",
                                  "weighted_shift_lula_pp"].iloc[0])),
                xytext=(16,-32),textcoords="offset points",
                color=RED,fontsize=9)
    a2.plot(cut["clock"],cut["paired_zones"],color=NAVY,
            lw=2,marker="o",markersize=5)
    a2.axvline(pd.Timestamp("2026-10-04 20:04:39"),color=RED,lw=1.5,ls="--")
    a2.set(xlabel="Horário usado para dividir os BUs por recebimento",
           ylabel="Zonas comparáveis")
    a2.grid(axis="y",alpha=.17)
    fig.autofmt_xdate()
    publish(fig,"03_mudanca_ja_acontecia_antes")
    display(Markdown("**O que observamos:** o contraste entre os BUs anteriores e "
        "posteriores ao corte já era positivo antes das 20h04. "
        "**O que isso enfraquece:** uma explicação em que toda a mudança teria "
        "começado precisamente na retomada. "
        "**O que não resolve:** se houve algum comportamento operacional adicional "
        "na publicação EA20."))
else:
    display(Markdown("Execute a **opção W** para mostrar os controles em outros horários."))

## 5 — De onde vieram as diferenças em relação à previsão por município?

As fases T/V criaram uma previsão para os BUs tardios com base nos recebidos antes de 20h04. O gráfico abaixo apresenta os **erros desse modelo em números de votos de Lula**.

**Crucial:** "votos acima do previsto" **não** quer dizer "votos falsos", "votos acrescentados" ou "votos sem BU". São erros de previsão; nosso objetivo é verificar onde ocorrem e se há uma explicação territorial.

In [ ]:
if v:
    cats=v["categories"]
    order=[
        ("MESMA_ZONA_COM_PRE","Zonas que já tinham BUs"),
        ("MUNICIPIO_SEM_PRE","Municípios sem BUs anteriores"),
        ("ZONA_SEM_PRE","Zonas novas em municípios conhecidos"),
    ]
    labels=[item[1] for item in order]
    errors=[cats[item[0]]["municipal_gap_votes"] for item in order]
    fig,ax=plt.subplots(figsize=(11,4))
    bars=ax.barh(labels,errors,color=[TEAL,TEAL,RED],height=.62)
    ax.axvline(0,color=GREY,lw=1)
    ax.invert_yaxis()
    ax.set(title="A maior parte do erro do modelo aparece em zonas já conhecidas",
           xlabel="Votos de Lula observados menos previstos (não são votos irregulares)")
    ax.grid(axis="x",alpha=.15)
    for bar,val in zip(bars,errors):
        ax.text(val+(6000 if val>=0 else -6000),
                bar.get_y()+bar.get_height()/2,
                f"{val:+,.0f}".replace(",","."),va="center",
                ha=("left" if val>=0 else "right"),fontsize=10)
    publish(fig,"04_decomposicao_do_erro_de_previsao")
    display(Markdown("**O que observamos:** o erro não se concentra "
        "exclusivamente em municípios que apareceram tarde. "
        "**Próximo teste:** distinguir a composição das seções dentro das zonas "
        "dos possíveis mecanismos de processamento dos resultados."))
else:
    display(Markdown("Execute a **opção V** para construir esta decomposição."))

## 6 — A prova operacional mais importante: recebido versus publicado

Este é o teste da opção **Y**. Para cada captura nacional EA20, comparamos quantos votos já constavam como recebidos no BWEB até aquele momento.

**Sob uma hipótese específica** — mesmo universo de votos, relógios compatíveis e recebimento anterior à publicação — o publicado não pode ultrapassar o recebido até aquele instante.

- **Diferença negativa:** o modelo simples não fecha naquele horário; precisamos investigar relógio, significado do registro ou processamento.
- **Diferença positiva:** a condição necessária foi atendida, mas não sabemos quais urnas estavam em qual lote.
- **Nunca:** declarar fraude apenas por uma diferença temporal sem certificar a equivalência dos registros.

In [ ]:
if y:
    timeline=checked_csv(y["output_files"]["timeline"])
    timeline["datetime"]=pd.to_datetime(timeline["ea20_brt"])
    pairs=[
        ("valid_received_minus_published","Votos válidos",NAVY),
        ("lula_received_minus_published","Lula",TEAL),
        ("flavio_received_minus_published","Flávio Bolsonaro",AMBER),
    ]
    fig,ax=plt.subplots(figsize=(11,5))
    for key,name,color in pairs:
        ax.plot(timeline["datetime"],
                timeline[key]/1_000_000,label=name,color=color,lw=1.9)
    ax.axhline(0,color=RED,lw=1.3,ls="--",
               label="Limite da hipótese: publicado ≤ recebido")
    ax.axvline(pd.Timestamp("2026-10-04 20:04:39"),
               lw=1.2,ls=":",color=GREY)
    ax.set(title="O que já estava recebido quando o resultado foi publicado?",
           xlabel="Horário declarado da captura EA20",
           ylabel="Recebidos menos publicados (milhões de votos)")
    ax.grid(axis="y",alpha=.15)
    ax.legend(frameon=False,ncol=2)
    fig.autofmt_xdate()
    publish(fig,"05_recebimento_vs_publicacao")
    count=y["baseline_offsets"]["snapshot_violations"]
    conclusion=("**O modelo temporal apresentou incompatibilidades em "
                +str(count)+" capturas.** Ainda é necessário conferir a "
                "semântica e sincronização dos relógios."
                if count else
                "**A condição agregada foi satisfeita nas capturas observadas.** "
                "Isso não prova que cada BU tenha sido publicado depois de seu "
                "recebimento, nem exclui outras hipóteses.")
    display(Markdown(conclusion))
else:
    display(Markdown("**Teste operacional pendente:** a opção Y ainda não "
      "produziu um resultado local verificado. Execute Y pelo menu BAT e "
      "abra novamente este painel. Não vamos preencher este gráfico "
      "com valores hipotéticos."));
    fig,ax=plt.subplots(figsize=(11,2.8))
    ax.axis("off")
    ax.text(.5,.62,"Recebido ≠ publicado",ha="center",va="center",
            fontsize=23,color=NAVY,weight="bold",transform=ax.transAxes)
    ax.text(.5,.32,"A evidência operacional será exibida após a fase Y.",
            ha="center",va="center",fontsize=12,color=GREY,transform=ax.transAxes)
    publish(fig,"05_teste_operacional_pendente")

## 7 — Quais hipóteses resistiram e o que ainda não demonstramos?

O papel do investigador não é colecionar anomalias. É **reduzir as explicações possíveis**, documentar refutações e indicar o teste que mudaria nossa conclusão. Esta é a situação atual da investigação, não um veredicto político.

In [ ]:
findings=[
    ("A linha acumulada é muito lisa",
     "Observado na série EA20",
     "Pode surgir de acumulação + sequência de volumes; oráculo é circular",
     "Não demonstra adulteração"),
    ("Os votos tardios têm composição distinta",
     "Observado em EA20 e BWEB, por métricas diferentes",
     "Geografia, seções e horário de recepção influenciam",
     "Ainda falta vincular BU a lote publicado"),
    ("O gradiente começou precisamente às 20h04",
     "Não sustentado pelos cortes W",
     "Já havia diferença antes, embora coortes mudem",
     "Não elimina outros eventos operacionais"),
    ("Votos divulgados sem terem sido recebidos",
     "Teste Y realizado" if y else "Pendente de execução da fase Y",
     "Exige relógios e conceitos compatíveis",
     "Sem identidade de seção, não é prova de alteração"),
    ("BU individual diverge do resultado oficial",
     "Não demonstrado por estas fases de curva",
     "Requer auditoria por seção e validação de autenticidade",
     "Sem conclusão de fraude a partir destas curvas"),
]
df=pd.DataFrame(findings,columns=[
    "Pergunta","Situação nos dados","Explicação/controle","Limite da conclusão"])
display(df)
display(Markdown(
    "**Conclusão provisória:** existem padrões de recebimento e divulgação "
    "que merecem investigação operacional e documentação pública. "
    "Os resultados aqui examinados **não constituem demonstração de fraude "
    "eleitoral**. Para avançar, nossa prioridade é o confronto por seção "
    "entre o BU verificável e o que foi efetivamente publicado, ou o acesso "
    "a registros temporais que permitam esse vínculo."))
print("Figuras para divulgação (PNG):",EXPORT)
print("Apenas resultados locais certificados foram utilizados.");


## 8 — Como divulgar sem induzir uma conclusão que os dados não sustentam

### Roteiro popular sugerido

> **Estamos auditando os números da apuração de 2026.** Descobrimos que a curva de divulgação era muito regular, embora os grupos de votos divulgados tivessem perfis diferentes. A diferença entre votos mais cedo e mais tarde já aparecia antes das 20h04. Isso não comprova fraude — nem dispensa investigação. O próximo passo é conferir se o que o sistema publicou era compatível com o que as urnas registraram e quando esses registros foram recebidos.

### Legenda que deve acompanhar qualquer imagem

**Fonte:** projeto TSE, primeiro turno presidencial de 2026, cálculos reproduzíveis sobre BWEB e capturas EA20. A visualização é descritiva; não identifica alteração de votos por si só. Consulte os scripts e hashes dos resultados da fase citada.

As figuras em PNG ficam salvas na pasta `H:\TSE\forensics\urnas\analysis\periodicidade_bweb_2026\divulgacao_publica_2026`.

**Próxima etapa editorial:** substituir qualquer mensagem de "fraude provada" por uma cadeia de prova verificável e publicar lado a lado os dados, o gráfico, a explicação alternativa e o documento que poderia decidir a controvérsia.